In [1]:
import pandas as pd
df_test=pd.read_csv('test.csv')
df_train=pd.read_csv('train.csv')

In [ ]:
from utils import download_images

# Fix: Create a fresh copy of the CSV data
dfcopy = pd.read_csv("test.csv")
image_links = dfcopy["image_link"].tolist()

# Let's test with a smaller subset first (first 10 images)
print(f"Total images to download: {len(image_links)}")

test_links = image_links[:75000]
try:
    download_images(test_links, download_folder="test_images_mapped_fr")
    print("✓ Successfully downloaded test images!")
except Exception as e:
    print(f"Error during download: {e}")
    print("Let's try an alternative approach...")

In [2]:
df_train.head()

,sample_id,catalog_content,image_link,price
0,33127,"Item Name: La Victoria Green Taco Sauce Mild, ...",https://m.media-amazon.com/images/I/51mo8htwTH...,4.89
1,198967,"Item Name: Salerno Cookies, The Original Butte...",https://m.media-amazon.com/images/I/71YtriIHAA...,13.12
2,261251,"Item Name: Bear Creek Hearty Soup Bowl, Creamy...",https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97
3,55858,Item Name: Judee’s Blue Cheese Powder 11.25 oz...,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34
4,292686,"Item Name: kedem Sherry Cooking Wine, 12.7 Oun...",https://m.media-amazon.com/images/I/41sA037+Qv...,66.49


In [6]:
from sklearn.model_selection import train_test_split

# Split the dataset into train and validation sets without stratification
train_data, val_data = train_test_split(df_train, test_size=0.1, random_state=42)

In [7]:
len(train_data)

67500

In [5]:
import os
from tensorflow.keras.models import load_model
import numpy as np

# Directory containing the .keras models
models_dir = "/Users/abhimanyu/Downloads"

# Load all .keras models from the directory
models = {}
for file in os.listdir(models_dir):
    if file.endswith(".keras"):
        model_path = os.path.join(models_dir, file)
        print(f"Loading model: {file}")
        models[file] = load_model(model_path)

print(f"Loaded {len(models)} models:")
for model_name in models.keys():
    print(f"- {model_name}")

Loading model: best_model_phase2 part 2(1).keras


ValueError: Input 0 of layer "stem_conv" is incompatible with the layer: expected axis -1 of input shape to have value 3, but received input with shape (None, 301, 301, 1)

In [ ]:
import cv2
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.imagenet_utils import preprocess_input
import requests
from PIL import Image
import io
import numpy as np

# Update the prepare_input_data function to properly handle image format conversion from grayscale to RGB and resize to expected dimensions

def prepare_input_data(image_url_or_path, target_size=(224, 224)):
    """
    Prepare input data for the models - converts images to RGB format and proper size

    Args:
        image_url_or_path: URL or local path to image
        target_size: Target size for the image (height, width)

    Returns:
        Preprocessed image array ready for model prediction
    """
    try:
        # Load image
        if image_url_or_path.startswith('http'):
            # Download image from URL
            response = requests.get(image_url_or_path)
            img = Image.open(io.BytesIO(response.content))
        else:
            # Load local image
            img = Image.open(image_url_or_path)

        # Convert to RGB if not already (handles grayscale, RGBA, etc.)
        if img.mode != 'RGB':
            img = img.convert('RGB')

        # Resize to target size
        img = img.resize(target_size)

        # Convert to numpy array
        img_array = image.img_to_array(img)

        # Add batch dimension
        img_array = np.expand_dims(img_array, axis=0)

        # Preprocess for models (normalize to 0-1 range)
        img_array = img_array / 255.0

        return img_array

    except Exception as e:
        print(f"Error processing image {image_url_or_path}: {str(e)}")
        return None

# Make predictions with each model
def make_ensemble_predictions(input_data):
    """
    Make predictions with all loaded models

    Args:
        input_data: Preprocessed image array

    Returns:
        Dictionary of predictions from each model
    """
    if input_data is None:
        print("No valid input data provided")
        return {}

    predictions = {}
    for model_name, model in models.items():
        try:
            pred = model.predict(input_data, verbose=0)
            predictions[model_name] = pred
            print(f"Prediction from {model_name}: {pred.flatten()}")
        except Exception as e:
            print(f"Error with model {model_name}: {str(e)}")
    return predictions

# Example usage (uncomment when you have actual input data):
# sample_image_url = "https://example.com/image.jpg"  # Replace with actual image URL
# processed_input = prepare_input_data(sample_image_url)
# ensemble_preds = make_ensemble_predictions(processed_input)

In [ ]:
# Process images from your dataset
def process_dataset_images(dataframe, image_column='image_path', sample_size=5):
    """
    Process images from your dataset for ensemble predictions
    
    Args:
        dataframe: DataFrame containing image paths/URLs
        image_column: Column name containing image paths or URLs
        sample_size: Number of samples to process (for testing)
    
    Returns:
        List of processed images and their corresponding data
    """
    results = []
    
    # Take a sample for testing
    sample_df = dataframe.head(sample_size)
    
    for idx, row in sample_df.iterrows():
        image_path_or_url = row[image_column]
        
        # Process the image
        processed_img = prepare_input_data(image_path_or_url)
        
        if processed_img is not None:
            # Make predictions with ensemble
            predictions = make_ensemble_predictions(processed_img)
            
            # Store results
            result = {
                'index': idx,
                'image_path': image_path_or_url,
                'predictions': predictions,
                'sample_id': row.get('sample_id', idx)  # Assuming there's a sample_id column
            }
            results.append(result)
            print(f"Processed image {idx}: {image_path_or_url}")
        else:
            print(f"Failed to process image {idx}: {image_path_or_url}")
    
    return results

# Example: Process validation set images
# Uncomment and adjust column names as needed:
# val_results = process_dataset_images(val_data, image_column='image_path', sample_size=3)

# Example: Process test set images for final predictions
# test_results = process_dataset_images(df_test, image_column='image_path', sample_size=10)

In [ ]:
# Separate validation images from existing train_images_mapped folder
import os
import shutil
from pathlib import Path

def separate_validation_images_from_existing():
    """
    Copy/move validation images from existing train_images_mapped folder
    """
    print("=" * 50)
    print("VALIDATION IMAGE SEPARATION FROM EXISTING FOLDER")
    print("=" * 50)
    
    # Paths
    source_folder = "train_images_mapped"
    val_folder = "validation_images"
    train_folder = "train_images_only"
    
    # Check if source folder exists
    if not os.path.exists(source_folder):
        print(f"✗ Source folder '{source_folder}' not found!")
        print("Please make sure your training images are in 'train_images_mapped' folder")
        return False
    
    # Create destination folders
    for folder in [val_folder, train_folder]:
        if not os.path.exists(folder):
            os.makedirs(folder)
            print(f"✓ Created folder: {folder}")
    
    # Get validation and training sample IDs
    val_sample_ids = set(val_data["sample_id"].astype(str).tolist())
    train_sample_ids = set(train_data["sample_id"].astype(str).tolist())
    
    print(f"Validation samples: {len(val_sample_ids)}")
    print(f"Training samples: {len(train_sample_ids)}")
    
    # Find and organize images
    val_copied = 0
    train_copied = 0
    not_found = 0
    
    # Process validation images
    print(f"\nSeparating validation images...")
    for sample_id in val_sample_ids:
        # Try different extensions
        for ext in ['.jpg', '.jpeg', '.png', '.webp']:
            source_path = os.path.join(source_folder, f"{sample_id}{ext}")
            if os.path.exists(source_path):
                dest_path = os.path.join(val_folder, f"{sample_id}{ext}")
                shutil.copy2(source_path, dest_path)  # copy2 preserves metadata
                val_copied += 1
                break
        else:
            not_found += 1
            if not_found <= 5:  # Show first 5 missing files
                print(f"  Missing: {sample_id}")
    
    # Process training images (remaining ones)
    print(f"Organizing remaining training images...")
    for sample_id in train_sample_ids:
        # Try different extensions
        for ext in ['.jpg', '.jpeg', '.png', '.webp']:
            source_path = os.path.join(source_folder, f"{sample_id}{ext}")
            if os.path.exists(source_path):
                dest_path = os.path.join(train_folder, f"{sample_id}{ext}")
                shutil.copy2(source_path, dest_path)
                train_copied += 1
                break
    
    # Create mapping files
    val_mapping = val_data[['sample_id', 'image_link', 'price']].copy()
    val_mapping.to_csv('validation_mapping.csv', index=False)
    
    train_mapping = train_data[['sample_id', 'image_link', 'price']].copy()
    train_mapping.to_csv('training_mapping.csv', index=False)
    
    print(f"\n✓ Image separation complete!")
    print(f"  - Validation images copied: {val_copied}/{len(val_sample_ids)}")
    print(f"  - Training images copied: {train_copied}/{len(train_sample_ids)}")
    print(f"  - Images not found: {not_found}")
    print(f"  - Validation folder: {val_folder}/")
    print(f"  - Training folder: {train_folder}/")
    print(f"  - Created: validation_mapping.csv, training_mapping.csv")
    
    return True

# Run the separation
success = separate_validation_images_from_existing()

if success:
    print(f"\n✅ Ready to run ensemble predictions on validation set!")
    print(f"Use 'validation_images' folder for local image processing.")

In [ ]:
# Run ensemble predictions on validation set using local images
def process_validation_images_for_ensemble(sample_size=None):
    """
    Process validation images for ensemble predictions using local separated images
    
    Args:
        sample_size: Number of validation samples to process (None for all)
    
    Returns:
        List of prediction results with actual vs predicted prices
    """
    print("=" * 50)
    print("VALIDATION ENSEMBLE PREDICTIONS")
    print("=" * 50)
    
    # Check if validation images folder exists
    val_folder = "validation_images"
    if not os.path.exists(val_folder):
        print(f"✗ Validation images folder '{val_folder}' not found!")
        print("Please run the image separation function first.")
        return []
    
    # Determine sample size
    if sample_size is None:
        sample_size = len(val_data)
    else:
        sample_size = min(sample_size, len(val_data))
    
    print(f"Processing {sample_size} validation images...")
    
    # Get sample of validation data
    val_sample = val_data.head(sample_size).copy()
    
    results = []
    successful_predictions = 0
    
    for idx, row in val_sample.iterrows():
        try:
            sample_id = str(row['sample_id'])
            
            # Find the image file (try different extensions)
            image_path = None
            for ext in ['.jpg', '.jpeg', '.png', '.webp']:
                potential_path = os.path.join(val_folder, f"{sample_id}{ext}")
                if os.path.exists(potential_path):
                    image_path = potential_path
                    break
            
            if image_path is None:
                print(f"✗ Image not found for sample_id: {sample_id}")
                continue
            
            # Process the image
            processed_img = prepare_input_data(image_path)
            
            if processed_img is not None:
                # Make predictions with ensemble
                predictions = make_ensemble_predictions(processed_img)
                
                if predictions:
                    # Ensemble the predictions
                    ensemble_pred = ensemble_predictions(predictions, method='average')
                    
                    # Store results
                    result = {
                        'sample_id': sample_id,
                        'actual_price': row['price'],
                        'ensemble_prediction': ensemble_pred.flatten()[0] if ensemble_pred is not None else None,
                        'individual_predictions': {k: v.flatten()[0] for k, v in predictions.items()},
                        'image_path': image_path,
                        'absolute_error': None,
                        'relative_error': None
                    }
                    
                    # Calculate errors
                    if result['ensemble_prediction'] is not None:
                        result['absolute_error'] = abs(result['actual_price'] - result['ensemble_prediction'])
                        result['relative_error'] = result['absolute_error'] / result['actual_price'] * 100
                    
                    results.append(result)
                    successful_predictions += 1
                    
                    if successful_predictions % 10 == 0:
                        print(f"✓ Processed {successful_predictions}/{sample_size} images...")
                
            else:
                print(f"✗ Failed to process image: {image_path}")
                
        except Exception as e:
            print(f"✗ Error processing sample_id {row['sample_id']}: {str(e)}")
    
    print(f"\n✅ Successfully processed {successful_predictions}/{sample_size} validation images")
    
    # Calculate summary statistics
    if results:
        errors = [r['absolute_error'] for r in results if r['absolute_error'] is not None]
        rel_errors = [r['relative_error'] for r in results if r['relative_error'] is not None]
        
        print(f"\n📊 VALIDATION PERFORMANCE SUMMARY:")
        print(f"  - Mean Absolute Error: ${np.mean(errors):.2f}")
        print(f"  - Median Absolute Error: ${np.median(errors):.2f}")
        print(f"  - Mean Relative Error: {np.mean(rel_errors):.1f}%")
        print(f"  - Median Relative Error: {np.median(rel_errors):.1f}%")
    
    return results

# Process a small sample first to test
print("Testing with 10 validation samples...")
val_results = process_validation_images_for_ensemble(sample_size=10)

# Display sample results
if val_results:
    print(f"\n📋 SAMPLE RESULTS:")
    for i, result in enumerate(val_results[:5]):  # Show first 5 results
        print(f"  Sample {i+1} (ID: {result['sample_id']}):")
        print(f"    Actual: ${result['actual_price']:.2f}")
        print(f"    Predicted: ${result['ensemble_prediction']:.2f}")
        print(f"    Error: ${result['absolute_error']:.2f} ({result['relative_error']:.1f}%)")
        print()

In [ ]:
# Ensemble predictions from multiple models
def ensemble_predictions(predictions_dict, method='average', weights=None):
    """
    Ensemble predictions from multiple models
    
    Args:
        predictions_dict: Dictionary with model names as keys and predictions as values
        method: 'average', 'weighted_average', or 'median'
        weights: List of weights for weighted average (should sum to 1)
    
    Returns:
        Ensembled predictions
    """
    if not predictions_dict:
        return None
    
    # Convert predictions to numpy arrays
    pred_arrays = [np.array(pred) for pred in predictions_dict.values()]
    
    if method == 'average':
        return np.mean(pred_arrays, axis=0)
    elif method == 'weighted_average' and weights:
        if len(weights) != len(pred_arrays):
            raise ValueError("Number of weights must match number of models")
        weighted_sum = sum(w * pred for w, pred in zip(weights, pred_arrays))
        return weighted_sum
    elif method == 'median':
        return np.median(pred_arrays, axis=0)
    else:
        return np.mean(pred_arrays, axis=0)  # Default to average

# Example usage:
# ensemble_result = ensemble_predictions(ensemble_preds, method='average')
# print(f"Ensemble prediction: {ensemble_result}")

# For weighted ensemble (adjust weights as needed):
# weights = [0.4, 0.3, 0.3]  # Adjust based on model performance
# weighted_ensemble = ensemble_predictions(ensemble_preds, method='weighted_average', weights=weights)